# Watermarks and Late-Arriving Data: Bounding Streaming State Memory

**Module**: 09_streaming  
**Notebook**: 04_watermarks_and_late_data.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How watermarks bound state store memory during streaming aggregations and drop late-arriving events.
- **Why does this topic exist?**: Without watermarks, streaming aggregations maintain historical state indefinitely, eventually crashing with OOM.
- **What problem does it solve?**: State store growing unbounded on high-velocity event streams.
- **Where is it used in real-world Data Engineering?**: Sessionizing user clickstreams and windowed IoT metrics.

---

## 2. Watermark Mechanics
`Watermark = MaxEventTimeSeen - AllowedDelay`
- Events with `event_time >= Watermark` are accepted into the aggregation window.
- Events with `event_time < Watermark` are discarded as too late.
- State for windows older than the watermark is cleared from memory.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Watermarks_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Define streaming query with 10-minute watermark and 5-minute tumbling window
rate = spark.readStream.format("rate").option("rowsPerSecond", 2).load()

windowed = (
    rate
    .withWatermark("timestamp", "10 seconds")
    .groupBy(F.window("timestamp", "5 seconds"))
    .count()
)

print("Streaming plan with Watermark:")
windowed.explain()

---

## Experiment: Running Watermarked Aggregation to Memory Sink

- **1. What are we testing?**: Run windowed aggregation with watermark for 6 seconds.
- **2. What do we expect?**: Computes window counts and prunes expired windows.

In [ ]:
q = windowed.writeStream.format("memory").queryName("window_sink").outputMode("update").start()
time.sleep(6)
spark.sql("SELECT window.start, window.end, count FROM window_sink ORDER BY window.start DESC LIMIT 5").show(truncate=False)
q.stop()

**3. What happened?**: Output windows emitted incrementally.  
**4. Why did it happen?**: Update mode emits changed window counts as rows arrive.  
**5. Practical Engineering Lesson**: Always define withWatermark() on all stateful streaming aggregations.

---

## Practical Exercises

### Exercise 1 (Beginner): Watermark Syntax
**Problem Statement**: Write withWatermark on timestamp column for 1 hour.  
**Input / Context**: code

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Tumbling vs Sliding Window
**Problem Statement**: Explain window('timestamp','10 minutes','5 minutes').  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): State Eviction Inspection
**Problem Statement**: Check state memory in streaming query progress.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Watermark Syntax):
```python
print('df.withWatermark("timestamp","1 hour")')
```

#### Solution 2 (Tumbling vs Sliding Window):
```python
print('10-minute window sliding every 5 minutes (sliding window).')
```

#### Solution 3 (State Eviction Inspection):
```python
print('q.lastProgress["stateOperators"]')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Watermark Purpose**: Purges old state from RAM to prevent OOM.
- **Late Data**: Data older than watermark is dropped.

### Essential Technical Terms
- `Watermark`: MaxEventTime - AllowedDelay threshold.
- `State Store`: In-memory or RocksDB state repository.

### Common Engineering Mistakes
- **Omitting Watermark on GroupBy**: State store grows indefinitely until executor crashes.

### Interview Questions on This Topic
- **Q: What problem does watermarking solve in Structured Streaming?**
  - *A*: It bounds state memory by specifying how long to wait for late data before purging historical aggregate windows from the state store.

### What to Study Next
Open 05_checkpoints_and_fault_tolerance.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')